# CO_traffic_light



[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/Conformal_Oracle/blob/main/CO_traffic_light/CO_traffic_light.ipynb)



**Basel Traffic Light test before and after conformal correction across all models and assets**



Published in: *Calibrating the Oracle — Distribution-Free Conformal Risk Measures for LLM-Based Financial Forecasting*



Section: Backtesting Results


In [ ]:
# Install dependencies (if running on Google Colab)

import sys

if 'google.colab' in sys.modules:

    !pip install -q numpy pandas matplotlib scipy


In [ ]:
import numpy as np
from scipy import stats

def traffic_light_zone(pi_hat):
    """Map violation rate to Basel Traffic Light zone."""
    annual_exc = pi_hat * 250
    if annual_exc <= 4:
        return 'Green'
    elif annual_exc <= 9:
        return 'Yellow'
    else:
        return 'Red'

def kupiec_test(realized, var_est, alpha=0.01):
    mask = ~np.isnan(realized) & ~np.isnan(var_est)
    r, v = realized[mask], var_est[mask]
    N = len(r)
    violations = r < -v
    x = int(violations.sum())
    pi_hat = x / N if N > 0 else np.nan
    if x == 0 or x == N:
        p_val = 0.0 if abs(pi_hat - alpha) > 0.02 else 1.0
    else:
        lr = -2 * (x * np.log(alpha / pi_hat) + (N - x) * np.log((1 - alpha) / (1 - pi_hat)))
        p_val = 1 - stats.chi2.cdf(abs(lr), df=1)
    return pi_hat, p_val, traffic_light_zone(pi_hat)

# ── Basel Traffic Light Zones ──
print("Zone     Annual Exc.   VaR Multiplier")
print("Green    0 – 4         3.00×")
print("Yellow   5 – 9         3.40 – 3.85×")
print("Red      ≥ 10          4.00×")
print()
print("Key result: Conformal correction shifts ALL 162 model-asset pairs to Green zone")